# Periodic and non-periodic FMM comparison

Set `USE_CUBOIDS` in the first code cell to compare either point dipoles or uniformly magnetised cuboids on the same 1,000 source/target centres. The notebook performs direct reference evaluations and an FMM parameter sweep:

1. **Direct, non-periodic:** exact all-to-all interactions in the central cell.
2. **Direct, periodic:** exact all-to-all interactions with the central cell and its 26 nearest images.
3. **Standard FMM:** non-periodic approximations for orders 4, 6, and 8 at tree depths 3, 4, and 5.
4. **Periodic FMM:** the same order/depth sweep with periodic boundaries.

The field and geometry visualisations use order 6 and depth 3. Accuracy plots compare all nine order/depth combinations. Both direct references use the CUDA dense all-to-all plan. In cuboid mode, direct and list1 near-field interactions are exact cuboid-to-volume-averaged-cuboid tensors, while P2M and L2P deliberately use the ordinary point representations. The main expectation is simple: `standard FMM vs direct non-periodic` and `periodic FMM vs direct periodic` compare matching physics and should give the two smallest, broadly similar errors. Cross-boundary comparisons use different physics and should be substantially larger.

> **One limitation:** the direct periodic calculation includes 27 cells, while the periodic FMM uses the infinite zero-$k=0$ lattice convention. The moments have zero total moment to reduce the omitted direct-image tail, but this finite-versus-infinite difference can still contribute to the periodic matched error.

The notebook does not modify or rebuild cdfmm. It assumes the current Python module has already been compiled and installed into the selected kernel environment.

In [ ]:
import itertools
import time

import cdfmm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
try:
    from example_utils import draw_box_3d, finish_3d_axes
except ModuleNotFoundError:
    from examples.notebooks.example_utils import (
        draw_box_3d,
        finish_3d_axes,
    )

# Change this flag to switch the complete comparison geometry.
USE_CUBOIDS = True

GRID_AXIS = 10
N_SOURCES = GRID_AXIS**3
MU0 = 4.0 * np.pi * 1.0e-7
SATURATION_POLARISATION_T = 1.2
SATURATION_MAGNETISATION = SATURATION_POLARISATION_T / MU0
CUBOID_SIDE = 10.0e-9
CUBOID_VOLUME = CUBOID_SIDE**3
CENTRE_SPACING = 3.0 * CUBOID_SIDE
DOMAIN_SIDE = GRID_AXIS * CENTRE_SPACING if USE_CUBOIDS else 1.0
FMM_ORDERS = [4, 6, 8]
TREE_DEPTHS = [3, 4, 5]
VISUAL_FMM_ORDER = 6
VISUAL_TREE_DEPTH = 3
BASELINE_IMAGE_RADIUS = 1       # 3^3 = 27 cells
SETUP_TOLERANCE = 1.0e-12
RNG_SEED = 20260825

if not cdfmm.cuda_dense_direct_available():
    raise RuntimeError(
        "This notebook requires a CUDA-enabled cdfmm build and "
        "an available CUDA device for the direct all-to-all reference"
    )

if cdfmm.cuda_full_available():
    EXECUTION_BACKEND = cdfmm.ExecutionBackend.CUDA_FULL
    EXECUTION_BACKEND_NAME = "CUDA full"
else:
    EXECUTION_BACKEND = cdfmm.ExecutionBackend.CPU_STATIC
    EXECUTION_BACKEND_NAME = "CPU static"

GEOMETRY_NAME = "cuboid" if USE_CUBOIDS else "point dipole"

print(
    f"geometry={GEOMETRY_NAME}; N={N_SOURCES:,}; "
    f"direct cells=1 and 27; "
    f"FMM orders={FMM_ORDERS}, depths={TREE_DEPTHS}; "
    f"visual case=({VISUAL_FMM_ORDER}, {VISUAL_TREE_DEPTH}); "
    f"FMM backend={EXECUTION_BACKEND_NAME}; "
    f"direct backend=CUDA dense"
)

## One shared magnetic problem

The 1,000 positions form a cell-centred $10^3$ lattice in $[-L/2,L/2)^3$. The same positions are both sources and targets in all four evaluations. Paired opposite random directions make $\sum_j m_j=0$ exactly up to round-off, reducing the omitted direct-image tail.

Point-dipole mode uses unit-magnitude total moments in dimensionless geometry. Cuboid mode uses non-overlapping 10 nm cubes at 30 nm centre spacing and a physically motivated saturation polarisation of 1.2 T. Because cdfmm consumes each cuboid's total moment, the uniform magnetisation is converted with $m=V M$ (equivalently $M=m/V$). Cuboid self-fields are finite and retained; only the singular central point-dipole self-pair is excluded.

In [ ]:
if N_SOURCES % 2 != 0:
    raise ValueError("Paired moment directions require an even source count")

rng = np.random.default_rng(RNG_SEED)
grid_axis = (
    (np.arange(GRID_AXIS, dtype=np.float64) + 0.5) / GRID_AXIS - 0.5
) * DOMAIN_SIDE
positions = np.array(
    np.meshgrid(grid_axis, grid_axis, grid_axis, indexing="ij")
).reshape(3, -1).T
half_directions = rng.normal(size=(N_SOURCES // 2, 3))
half_directions /= np.linalg.norm(half_directions, axis=1)[:, None]
moment_directions = np.concatenate((half_directions, -half_directions))
rng.shuffle(moment_directions, axis=0)

if USE_CUBOIDS:
    magnetisations = SATURATION_MAGNETISATION * moment_directions
    moments = CUBOID_VOLUME * magnetisations
    cuboid = cdfmm.CuboidSize(
        CUBOID_SIDE, CUBOID_SIDE, CUBOID_SIDE
    )
else:
    magnetisations = None
    moments = moment_directions.copy()
    cuboid = None
positions = np.ascontiguousarray(positions)
moments = np.ascontiguousarray(moments)
self_indices = np.arange(N_SOURCES, dtype=np.int32)

print("Total moment:", moments.sum(axis=0))
print("RMS total-moment magnitude:", np.sqrt(
    np.mean(np.sum(moments**2, axis=1))
))
if USE_CUBOIDS:
    print(f"Cube side: {CUBOID_SIDE * 1.0e9:.1f} nm")
    print(f"Centre spacing: {CENTRE_SPACING * 1.0e9:.1f} nm")
    print("RMS magnetisation magnitude:", np.sqrt(
        np.mean(np.sum(magnetisations**2, axis=1))
    ), "A/m")

## Visualise the fundamental cell and its 26 nearest images

Blue source centres are stored once in the fundamental cell. The arrows show dipole or magnetisation directions. Orange points show a small subset copied into the 26 neighbouring cells for visualisation only; the FMM does not replicate its particle arrays.

In [ ]:
def draw_cell_wireframe(axis, shift, side, color, alpha, linewidth):
    lower = np.asarray(shift, dtype=float) * side - 0.5 * side
    upper = lower + side
    corners = np.array(list(itertools.product(*zip(lower, upper))))
    for first, second in itertools.combinations(corners, 2):
        if np.count_nonzero(first != second) == 1:
            axis.plot(*zip(first, second), color=color, alpha=alpha, lw=linewidth)

fig = plt.figure(figsize=(10, 8))
axis = fig.add_subplot(111, projection="3d")
sample = np.linspace(0, N_SOURCES - 1, min(24, N_SOURCES), dtype=int)
for shift in itertools.product((-1, 0, 1), repeat=3):
    central = shift == (0, 0, 0)
    draw_cell_wireframe(
        axis, shift, DOMAIN_SIDE,
        color="black" if central else "0.55",
        alpha=1.0 if central else 0.35,
        linewidth=1.8 if central else 0.7,
    )
    displaced = positions[sample] + DOMAIN_SIDE * np.asarray(shift)
    axis.scatter(
        displaced[:, 0], displaced[:, 1], displaced[:, 2],
        s=13 if central else 5,
        color="tab:blue" if central else "tab:orange",
        alpha=0.9 if central else 0.18,
    )
axis.quiver(
    positions[sample, 0], positions[sample, 1], positions[sample, 2],
    moments[sample, 0], moments[sample, 1], moments[sample, 2],
    length=0.12 * DOMAIN_SIDE, normalize=True,
    color="tab:blue", linewidth=0.7,
)
axis.set(
    xlabel="x", ylabel="y", zlabel="z",
    title="Fundamental cell and the 26 explicitly traversed images",
)
axis.set_box_aspect((1, 1, 1))
fig.tight_layout()

## Compare non-periodic and periodic list1/list2 geometry

As in notebook 08, these plots draw three-dimensional interaction boxes around one corner leaf. A non-periodic list stops at the domain boundary. A periodic list wraps across that boundary. Periodic interactions are drawn at the location of their original node in the central tree—not at a displaced image location—and highlighted in orange or purple.

In [ ]:
def periodic_octree_stencils(level, target_coordinate):
    target = np.asarray(target_coordinate, dtype=int)
    list1 = [
        tuple(target + np.asarray(offset))
        for offset in itertools.product((-1, 0, 1), repeat=3)
    ]
    list1_set = set(list1)
    parent = target // 2
    list2 = []
    for parent_offset in itertools.product((-1, 0, 1), repeat=3):
        source_parent = parent + np.asarray(parent_offset)
        for child_bits in itertools.product((0, 1), repeat=3):
            source = tuple(2 * source_parent + np.asarray(child_bits))
            if source not in list1_set:
                list2.append(source)
    return list1, list2

def is_periodic_box(coordinate, boxes_per_axis):
    coordinate = np.asarray(coordinate)
    return bool(np.any(coordinate < 0) or np.any(coordinate >= boxes_per_axis))

def box_geometry(coordinate, boxes_per_axis):
    half_width = 0.5 * DOMAIN_SIDE / boxes_per_axis
    centre = (np.asarray(coordinate, dtype=float) + 0.5) * (
        DOMAIN_SIDE / boxes_per_axis
    ) - 0.5 * DOMAIN_SIDE
    return centre, half_width

def plot_stencil_3d(
    axis, entries, target, boxes_per_axis, title,
    central_colour, image_colour,
):
    draw_box_3d(
        axis, np.zeros(3), 0.5 * DOMAIN_SIDE, colour="black",
        linewidth=1.8, alpha=0.8, label="central domain",
    )
    central_label_available = True
    image_label_available = True
    for entry in entries:
        image_entry = is_periodic_box(entry, boxes_per_axis)
        displayed_entry = (
            np.mod(entry, boxes_per_axis) if image_entry else entry
        )
        centre, half_width = box_geometry(displayed_entry, boxes_per_axis)
        label = None
        if image_entry and image_label_available:
            label = "interaction through periodic image"
            image_label_available = False
        elif not image_entry and central_label_available:
            label = "central-tree interaction"
            central_label_available = False
        draw_box_3d(
            axis, centre, half_width,
            colour=image_colour if image_entry else central_colour,
            linewidth=1.0, alpha=0.65, label=label,
        )
    target_centre, target_half_width = box_geometry(target, boxes_per_axis)
    draw_box_3d(
        axis, target_centre, target_half_width, colour="tab:red",
        linewidth=2.8, alpha=1.0, label="target leaf",
    )
    finish_3d_axes(axis, f"{title}\n{len(entries)} interaction identities")
    axis.legend(loc="upper left", fontsize=8)

def compare_stencils(non_periodic, periodic, target, title, colours):
    figure = plt.figure(figsize=(15, 6.5))
    axes = [
        figure.add_subplot(121, projection="3d"),
        figure.add_subplot(122, projection="3d"),
    ]
    plot_stencil_3d(
        axes[0], non_periodic, target, boxes_per_axis,
        f"Non-periodic {title}", colours[0], colours[1],
    )
    plot_stencil_3d(
        axes[1], periodic, target, boxes_per_axis,
        f"Periodic {title}", colours[0], colours[1],
    )
    figure.tight_layout()

boxes_per_axis = 2**VISUAL_TREE_DEPTH
target_box = np.array([0, 0, 0])
periodic_list1, periodic_list2 = periodic_octree_stencils(
    VISUAL_TREE_DEPTH, target_box
)
non_periodic_list1 = [
    entry for entry in periodic_list1
    if not is_periodic_box(entry, boxes_per_axis)
]
non_periodic_list2 = [
    entry for entry in periodic_list2
    if not is_periodic_box(entry, boxes_per_axis)
]

compare_stencils(
    non_periodic_list1, periodic_list1, target_box,
    "list1: exact P2P", ("tab:blue", "tab:orange"),
)
compare_stencils(
    non_periodic_list2, periodic_list2, target_box,
    "list2: M2L", ("tab:green", "tab:purple"),
)

list_counts = pd.DataFrame([
    dict(list="list1", non_periodic=len(non_periodic_list1),
         periodic=len(periodic_list1),
         periodic_images=len(periodic_list1) - len(non_periodic_list1)),
    dict(list="list2", non_periodic=len(non_periodic_list2),
         periodic=len(periodic_list2),
         periodic_images=len(periodic_list2) - len(non_periodic_list2)),
])
display(list_counts)

## Build the non-periodic and 27-cell direct references

The non-periodic reference contains the central $N^2$ interactions only. The periodic baseline sums over the nearest $3\times3\times3$ source cells, for $27N^2=27,000,000$ pair constructions.

Both modes construct and evaluate one FP64 `CudaDenseDirectPlan` per source image, then release it before the next image; this keeps only one six-component $N\times N$ tensor on the GPU instead of retaining a 27-cell tensor of roughly 1.2 GiB. Point mode uses exact point-dipole tensors, while cuboid mode uses exact cuboid-to-volume-averaged-cuboid tensors. The point-dipole central diagonal is excluded. Exact cuboid self-fields and every non-zero self-image are physical and retained.

In [ ]:
def evaluate_cuda_dense_image_sum(
    source_positions, target_positions, source_moments, side,
    image_radius, cuboid_size=None,
):
    field = np.zeros((len(target_positions), 3), dtype=np.float64)
    setup_seconds = 0.0
    evaluation_seconds = 0.0
    shifts = range(-image_radius, image_radius + 1)
    for image_shift in itertools.product(shifts, repeat=3):
        shifted_sources = np.ascontiguousarray(
            source_positions + side * np.asarray(image_shift)
        )
        plan_arguments = dict(
            source_positions=shifted_sources,
            target_positions=target_positions,
        )
        if cuboid_size is not None:
            plan_arguments.update(
                source_geometry=cdfmm.SourceGeometry.UNIFORM_CUBOID,
                target_geometry=(
                    cdfmm.TargetGeometry.VOLUME_AVERAGED_CUBOID
                ),
                source_sizes=[cuboid_size],
                target_sizes=[cuboid_size],
            )
        elif image_shift == (0, 0, 0):
            plan_arguments["target_source_indices"] = (
                self_indices.tolist()
            )

        start = time.perf_counter()
        plan = cdfmm.CudaDenseDirectPlan(**plan_arguments)
        setup_seconds += time.perf_counter() - start

        start = time.perf_counter()
        field += np.asarray(plan.evaluate(source_moments))
        evaluation_seconds += time.perf_counter() - start
        del plan
    return field, setup_seconds, evaluation_seconds

(
    H_direct_nonperiodic,
    direct_nonperiodic_setup_s,
    direct_nonperiodic_evaluation_s,
) = evaluate_cuda_dense_image_sum(
    positions, positions, moments, DOMAIN_SIDE, 0, cuboid
)
(
    H_direct_periodic,
    direct_periodic_setup_s,
    direct_periodic_evaluation_s,
) = evaluate_cuda_dense_image_sum(
    positions, positions, moments, DOMAIN_SIDE,
    BASELINE_IMAGE_RADIUS, cuboid,
)

print(f"Direct non-periodic setup:    {direct_nonperiodic_setup_s:.3f} s")
print(f"Direct non-periodic evaluate: {direct_nonperiodic_evaluation_s:.6f} s")
print(f"Direct periodic setup:        {direct_periodic_setup_s:.3f} s")
print(f"Direct periodic evaluate:     {direct_periodic_evaluation_s:.6f} s")

## Direct field: periodic versus non-periodic

The first two panels use the same colour scale. The third shows their difference. This difference is the field added by the 26 neighbouring source images; it is a change in the physical model, not an FMM error.

In [ ]:
field_plot_indices = np.flatnonzero(
    np.isclose(positions[:, 2], grid_axis[GRID_AXIS // 2])
)
field_plot_positions = positions[field_plot_indices]

def plot_boundary_comparison(non_periodic, periodic, method):
    difference = periodic - non_periodic
    component_limit = np.max(np.abs(np.concatenate([
        non_periodic[field_plot_indices, 2],
        periodic[field_plot_indices, 2],
    ])))
    difference_limit = np.max(np.abs(difference[field_plot_indices, 2]))

    figure, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    panels = [
        (non_periodic[field_plot_indices, 2],
         f"Non-periodic {method}: $H_z$", "coolwarm",
         -component_limit, component_limit, "$H_z$"),
        (periodic[field_plot_indices, 2],
         f"Periodic {method}: $H_z$", "coolwarm",
         -component_limit, component_limit, "$H_z$"),
        (difference[field_plot_indices, 2],
         fr"Periodic - non-periodic {method}: $\Delta H_z$",
         "coolwarm", -difference_limit, difference_limit, r"$\Delta H_z$"),
    ]
    for axis, (values, title, cmap, lower, upper, label) in zip(
        axes, panels
    ):
        scatter = axis.scatter(
            field_plot_positions[:, 0], field_plot_positions[:, 1],
            c=values, s=48, cmap=cmap, vmin=lower, vmax=upper,
        )
        axis.set(title=title, xlabel="x", ylabel="y", aspect="equal")
        figure.colorbar(scatter, ax=axis, label=label)
        axis.grid(True, alpha=0.25)
    figure.tight_layout()
    return difference

plot_boundary_comparison(
    H_direct_nonperiodic, H_direct_periodic,
    f"direct {GEOMETRY_NAME}",
)

## Sweep the standard and periodic FMM parameters

For each boundary condition, run all nine combinations of expansion order $p \in \{4,6,8\}$ and tree depth $d \in \{3,4,5\}$. Every run uses the same positions, total moments, spherical basis, precision, and execution backend. The non-periodic tree derives its root from the particle bounds. The periodic plan instead fixes its root to the explicit cell and wraps list interactions across its boundaries.

In point-dipole mode, the original particle indices exclude the singular central self-pair. In cuboid mode, source and target geometry select exact cuboid-to-cuboid list1 tensors and retain the finite self-field. `use_cuboid_p2m=False` and `use_cuboid_l2p=False` deliberately keep the far-field source and target representations as ordinary point expansions, isolating finite geometry to the near field as requested.

The resulting fields are cached by `(order, depth, periodic)`. The order-6, depth-3 fields are selected separately for the visualisations below.

In [ ]:
def relative_l2(reference, result):
    return np.linalg.norm(result - reference) / np.linalg.norm(reference)

def make_fmm_options(periodic, expansion_order, tree_depth):
    options = cdfmm.UniformFmmOptions()
    options.expansion_basis = cdfmm.ExpansionBasis.SPHERICAL
    options.expansion_order = expansion_order
    options.precision = cdfmm.StaticPrecision.FLOAT64
    options.tree.max_level = tree_depth
    options.backend = EXECUTION_BACKEND
    if USE_CUBOIDS:
        options.source_geometry = cdfmm.SourceGeometry.UNIFORM_CUBOID
        options.source_sizes = [cuboid]
        options.target_geometry = (
            cdfmm.TargetGeometry.VOLUME_AVERAGED_CUBOID
        )
        options.target_sizes = [cuboid]
        options.use_cuboid_p2m = False
        options.use_cuboid_l2p = False
    else:
        options.fixed_target_source_indices = self_indices.tolist()
    if periodic:
        options.periodic.enabled = True
        options.periodic.axes = [True, True, True]
        options.periodic.centre = cdfmm.Vec3(0.0, 0.0, 0.0)
        options.periodic.lengths = cdfmm.Vec3(
            DOMAIN_SIDE, DOMAIN_SIDE, DOMAIN_SIDE
        )
        options.periodic.convention = cdfmm.PeriodicConvention.ZeroK0
        options.periodic.setup_tolerance = SETUP_TOLERANCE
    return options

def run_fmm(periodic, expansion_order, tree_depth):
    start = time.perf_counter()
    fmm = cdfmm.UniformFmm(
        positions, positions,
        make_fmm_options(periodic, expansion_order, tree_depth),
    )
    setup_seconds = time.perf_counter() - start

    start = time.perf_counter()
    result = fmm.evaluate(moments)
    evaluation_seconds = time.perf_counter() - start
    field = np.asarray(result["H"]).copy()
    return field, setup_seconds, evaluation_seconds

fmm_fields = {}
accuracy_rows = []
for expansion_order in FMM_ORDERS:
    for tree_depth in TREE_DEPTHS:
        for periodic in (False, True):
            field, setup_seconds, evaluation_seconds = run_fmm(
                periodic, expansion_order, tree_depth
            )
            fmm_fields[(expansion_order, tree_depth, periodic)] = field
            reference = (
                H_direct_periodic if periodic else H_direct_nonperiodic
            )
            accuracy_rows.append(dict(
                order=expansion_order,
                tree_depth=tree_depth,
                boundary="periodic" if periodic else "non-periodic",
                relative_L2=relative_l2(reference, field),
                setup_s=setup_seconds,
                evaluation_s=evaluation_seconds,
            ))
            print(
                f"{'Periodic' if periodic else 'Standard':12s} FMM "
                f"p={expansion_order}, depth={tree_depth}: "
                f"setup={setup_seconds:.3f} s, "
                f"evaluate={evaluation_seconds:.6f} s"
            )

accuracy_results = pd.DataFrame(accuracy_rows)
visual_key = (VISUAL_FMM_ORDER, VISUAL_TREE_DEPTH)
H_fmm_nonperiodic = fmm_fields[(*visual_key, False)]
H_fmm_periodic = fmm_fields[(*visual_key, True)]

display(accuracy_results)

## FMM field: periodic versus non-periodic

This repeats the same three-panel field plot for the order-6, depth-3 FMM runs. As above, the third panel is a boundary-condition difference rather than an approximation error.

In [ ]:
plot_boundary_comparison(
    H_fmm_nonperiodic, H_fmm_periodic,
    (f"{GEOMETRY_NAME} FMM "
     f"(p={VISUAL_FMM_ORDER}, depth={VISUAL_TREE_DEPTH})"),
)

## The five comparisons

Every row reports one relative L2 field difference for the order-6, depth-3 visual case. Green rows compare matching boundary conditions and should be smallest. Orange rows compare different boundary conditions and should be much larger. The grey direct-versus-direct row measures how much the boundary condition changes the field; it is not an approximation error.

In [ ]:
comparisons = pd.DataFrame([
    dict(
        comparison="Direct periodic vs direct non-periodic",
        meaning="different boundary conditions",
        relative_L2=relative_l2(H_direct_nonperiodic, H_direct_periodic),
        colour="0.55",
    ),
    dict(
        comparison="Standard FMM vs direct non-periodic",
        meaning="matching physics: expected small",
        relative_L2=relative_l2(H_direct_nonperiodic, H_fmm_nonperiodic),
        colour="tab:green",
    ),
    dict(
        comparison="Standard FMM vs direct periodic",
        meaning="different boundary conditions",
        relative_L2=relative_l2(H_direct_periodic, H_fmm_nonperiodic),
        colour="tab:orange",
    ),
    dict(
        comparison="Periodic FMM vs direct non-periodic",
        meaning="different boundary conditions",
        relative_L2=relative_l2(H_direct_nonperiodic, H_fmm_periodic),
        colour="tab:orange",
    ),
    dict(
        comparison="Periodic FMM vs direct periodic",
        meaning="matching physics: expected small",
        relative_L2=relative_l2(H_direct_periodic, H_fmm_periodic),
        colour="tab:green",
    ),
])
display(comparisons[["comparison", "meaning", "relative_L2"]])

## Accuracy across expansion order and tree depth

The table and plots below compare matching physics only: standard FMM against the direct non-periodic reference, and periodic FMM against the direct periodic reference. Each curve holds tree depth fixed while expansion order changes. This makes both the order convergence and the effect of refining the tree visible.

In [ ]:
accuracy_table = accuracy_results.pivot(
    index=["order", "tree_depth"],
    columns="boundary",
    values="relative_L2",
).rename_axis(columns=None)
display(accuracy_table)

figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for axis, boundary, title in zip(
    axes,
    ("non-periodic", "periodic"),
    ("Standard FMM vs direct", "Periodic FMM vs direct"),
):
    boundary_results = accuracy_results[
        accuracy_results["boundary"] == boundary
    ]
    for tree_depth in TREE_DEPTHS:
        depth_results = boundary_results[
            boundary_results["tree_depth"] == tree_depth
        ].sort_values("order")
        axis.plot(
            depth_results["order"],
            depth_results["relative_L2"],
            marker="o",
            linewidth=2.0,
            label=f"depth {tree_depth}",
        )
    axis.set(
        title=title,
        xlabel="expansion order",
        ylabel="relative L2 field error",
        xticks=FMM_ORDERS,
    )
    axis.set_yscale("log")
    axis.grid(True, which="both", alpha=0.3)
    axis.legend()
figure.suptitle(
    f"{GEOMETRY_NAME.title()} FMM accuracy by order and tree depth"
)
figure.tight_layout()

## All four fields on one colour scale

These panels show the same $H_z$ slice for both direct references and the order-6, depth-3 FMM visual case. Matching direct/FMM pairs should look alike; switching the boundary condition may visibly change the field.

In [ ]:
field_results = [
    ("Direct, non-periodic", H_direct_nonperiodic),
    ("Direct, periodic", H_direct_periodic),
    (f"Standard {GEOMETRY_NAME} FMM, p={VISUAL_FMM_ORDER}, "
     f"depth={VISUAL_TREE_DEPTH}", H_fmm_nonperiodic),
    (f"Periodic {GEOMETRY_NAME} FMM, p={VISUAL_FMM_ORDER}, "
     f"depth={VISUAL_TREE_DEPTH}", H_fmm_periodic),
]
field_limit = np.max(np.abs(np.concatenate([
    field[field_plot_indices, 2] for _, field in field_results
])))
figure, axes = plt.subplots(2, 2, figsize=(11, 9))
for axis, (title, field) in zip(axes.flat, field_results):
    scatter = axis.scatter(
        field_plot_positions[:, 0], field_plot_positions[:, 1],
        c=field[field_plot_indices, 2], s=48, cmap="coolwarm",
        vmin=-field_limit, vmax=field_limit,
    )
    axis.set(title=f"{title}: $H_z$", xlabel="x", ylabel="y",
             aspect="equal")
    figure.colorbar(scatter, ax=axis, label="$H_z$")
    axis.grid(True, alpha=0.25)
figure.tight_layout()

In [ ]:
figure, axis = plt.subplots(figsize=(10, 5.5))
axis.barh(
    comparisons["comparison"], comparisons["relative_L2"],
    color=comparisons["colour"],
)
axis.set_xscale("log")
axis.set(
    xlabel="relative L2 difference",
    title=(
        f"Five {GEOMETRY_NAME} comparisons for "
        f"p={VISUAL_FMM_ORDER}, depth={VISUAL_TREE_DEPTH}"
    ),
)
axis.grid(True, axis="x", alpha=0.3)
figure.tight_layout()

## Interpretation checklist

- The two green comparisons use matching physics for the order-6, depth-3 visual case and should have the smallest errors.
- The orange comparisons mix periodic and non-periodic physics and should be substantially larger.
- The grey direct-versus-direct value measures the physical effect of adding periodic images; it is not a numerical error.
- In cuboid mode, direct and FMM list1 interactions are exact cuboid-to-cuboid fields, including the finite central self-field; only the far field uses point P2M and point L2P.
- The two green errors should be broadly similar. If the periodic one is larger, remember that its direct reference contains 27 cells while the periodic FMM contains the infinite zero-$k=0$ lattice.
- In the sweep plots, compare curves at fixed depth to see order convergence, and compare curves at fixed order to see the effect of tree refinement.